# Quantum Canary

Check your qubits before a job, between the jobs of a long workflow, and after.

Install: `pip install quantum-canary` (from a clone of this repository: `pip install -e .`).

In [1]:
# %pip install -e .

In [2]:
import warnings
warnings.filterwarnings("ignore")

import quantum_canary as canary

## Backend

This uses `fake_fez`, a local copy of IBM's 156-qubit Heron device with its real calibration and noise. It needs no account. Set `USE_IBM = True` to use your saved IBM account instead.

In [3]:
USE_IBM = False

if USE_IBM:
    from qiskit_ibm_runtime import QiskitRuntimeService
    backend = QiskitRuntimeService().backend("ibm_kingston")
else:
    from qiskit_ibm_runtime.fake_provider import FakeProviderForBackendV2
    backend = FakeProviderForBackendV2().backend("fake_fez")
backend.name

'fake_fez'

## 1. Before your job

One short job measures T1, T2, T2\*, the frequency offset and the gate error of each qubit, with error bars, a fit check (chi2/dof) and a health verdict with its reason.

In [4]:
before = canary.check(backend, [13, 14])
before.print()
before.save("before.json")

qubit T1               T2               T2*              dw/2pi (kHz)     eps                 fit                       health
------------------------------------------------------------------------------------------------------------------------------
13    259 ± 12 us      203 ± 20 us      193 ± 12 us      +0.121 ± 0.062   1.60e-04 ± 1e-05    ok                        GREEN
14    230 ± 11 us      43.3 ± 6.9 us    44 ± 2.7 us      -0.137 ± 0.24    1.62e-04 ± 1e-05    echo 5.1 > 4.4            YELLOW (fit)

dynamical decoupling: not needed on these qubits

Fit     chi2/dof of each probe against its limit. On a qubit that follows the model
        each probe goes over its limit 1.25% of the time; when it does, that probe's
        numbers may be wrong.
Health  GREEN   every fit converged and passed, and T2* is within 20% of T2
        YELLOW  a fit failed its limit, or T2* differs from T2 by more than 20%
                (slow frequency noise, which dynamical decoupling refocuses)
    

In [5]:
q = before[13]
q.t1, q.t2, q.t2_star, q.health, q.fit

(0.0002593750609978592,
 0.00020298445644860815,
 0.00019314450149821247,
 'GREEN',
 [])

## 2. Between jobs (workflows longer than about 30 minutes)

The first `check()` is the baseline; call it again between your jobs. Each later check is compared with the baseline, with the |z| limit raised for the number of checks.

In [6]:
mon = canary.Monitor(backend, [13, 14])
mon.check()   # baseline
# ... your job ...
mon.check()   # between jobs
mon.print()
mon.changed

1 check against the baseline: changed = |z| > 2.00, so that the chance of any false alarm over all of them is that of one check at |z| > 2

check 1 (2026-10-15T23:43:40)
qubit before   after         z T1     z T2     z dw    z eps   changed
----------------------------------------------------------------------
13    YELLOW   GREEN        -0.26    +0.39    +2.37    +0.46   -
14    GREEN    GREEN        +0.17    +0.54    -0.53    +0.02   -

changed = |z| > 2.00 after a Holm correction over each qubit's four parameters; z uses both checks' error bars
stable: nothing changed beyond the error bars


[]

## 3. After your job

Check again and compare with the first check.

In [7]:
after = canary.check(backend, [13, 14])
drift = canary.compare(canary.load("before.json"), after)
drift.print()
drift.changed

qubit before   after         z T1     z T2     z dw    z eps   changed
----------------------------------------------------------------------
13    GREEN    GREEN        +1.06    +0.30    -2.07    +0.66   -
14    YELLOW   GREEN        +0.47    +0.79    +0.76    +0.09   -

changed = |z| > 2.00 after a Holm correction over each qubit's four parameters; z uses both checks' error bars
stable: nothing changed beyond the error bars


[]

## No hardware at all

Simulated qubits, with their true values kept for comparison.

In [8]:
sim = canary.simulate(2, regime="ideal")
sim.print()
[(q.t1, q.truth["T1_s"]) for q in sim]

qubit T1               T2               T2*              dw/2pi (kHz)     eps                 fit                       health
------------------------------------------------------------------------------------------------------------------------------
0     190 ± 9.1 us     98 ± 8.8 us      83.4 ± 5.6 us    -4.15 ± 0.16     3.07e-04 ± 2e-05    ok                        GREEN
1     114 ± 5.4 us     91.6 ± 6.7 us    98.8 ± 6.5 us    +6.14 ± 0.12     5.01e-04 ± 3e-05    ok                        GREEN

dynamical decoupling advised on qubit 0: |Δω|·T2* = 2.2 rad > 1
dynamical decoupling advised on qubit 1: |Δω|·T2* = 3.8 rad > 1

Fit     chi2/dof of each probe against its limit. On a qubit that follows the model
        each probe goes over its limit 1.25% of the time; when it does, that probe's
        numbers may be wrong.
Health  GREEN   every fit converged and passed, and T2* is within 20% of T2
        YELLOW  a fit failed its limit, or T2* differs from T2 by more than 20%
         

[(0.00019043314863678617, 0.00018899296052325023),
 (0.00011391130105392075, 0.00011799078236726674)]